# Harmonizing the Data of your Data - Kaggle Submission

SDRF Metadata Extraction from Scientific Publications using LLM

In [ ]:
import json
import pandas as pd
import re
from pathlib import Path
import os

In [ ]:
# Kaggle paths
INPUT_DIR = Path('/kaggle/input/harmonizing-the-data-of-your-data')
OUTPUT_DIR = Path('/kaggle/working')

# Local paths (for testing)
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../data/raw')
    OUTPUT_DIR = Path('../submissions')

In [ ]:
# Load the baseline prompt
with open(INPUT_DIR / 'BaselinePrompt.txt', 'r') as f:
    BASELINE_PROMPT = f.read()

print(f"Prompt length: {len(BASELINE_PROMPT)} chars")

In [ ]:
# Load sample submission
sample_sub = pd.read_csv(INPUT_DIR / 'SampleSubmission.csv')
print(f"Submission shape: {sample_sub.shape}")
print(f"Columns: {sample_sub.columns.tolist()[:10]}...")

In [ ]:
# Get unique PXD IDs from submission
pxd_ids = sample_sub['PXD'].unique()
print(f"PXD IDs in submission: {pxd_ids}")

In [ ]:
# Load test manuscripts
test_dir = INPUT_DIR / 'Test PubText' / 'Test PubText'
test_data = {}

for json_file in test_dir.glob('PXD*_PubText.json'):
    pxd_id = json_file.stem.replace('_PubText', '')
    with open(json_file, 'r', encoding='utf-8') as f:
        test_data[pxd_id] = json.load(f)

print(f"Loaded {len(test_data)} test manuscripts")

In [ ]:
class SDRFExtractor:
    """Extract SDRF metadata using regex patterns."""
    
    PATTERNS = {
        'Characteristics[Organism]': [
            (r'Homo sapiens', 'Homo sapiens'),
            (r'human', 'Homo sapiens'),
            (r'Mus musculus', 'Mus musculus'),
            (r'mouse', 'Mus musculus'),
            (r'Rattus norvegicus', 'Rattus norvegicus'),
            (r'rat', 'Rattus norvegicus'),
            (r'Saccharomyces cerevisiae', 'Saccharomyces cerevisiae'),
            (r'yeast', 'Saccharomyces cerevisiae'),
            (r'Escherichia coli', 'Escherichia coli'),
            (r'E\.? coli', 'Escherichia coli'),
        ],
        'Characteristics[CellLine]': [
            (r'HEK293T?', None),
            (r'HeLa', None),
            (r'U2OS', None),
            (r'MCF-?7', None),
            (r'A549', None),
            (r'K562', None),
            (r'Jurkat', None),
        ],
        'Characteristics[CleavageAgent]': [
            (r'[Tt]rypsin', 'Trypsin'),
            (r'Lys-?C', 'Lys-C'),
            (r'chymotrypsin', 'Chymotrypsin'),
            (r'Glu-?C', 'Glu-C'),
        ],
        'Characteristics[AlkylationReagent]': [
            (r'iodoacetamide|IAA', 'iodoacetamide'),
            (r'chloroacetamide|CAA', 'chloroacetamide'),
        ],
        'Characteristics[ReductionReagent]': [
            (r'DTT|dithiothreitol', 'DTT'),
            (r'TCEP', 'TCEP'),
        ],
        'Comment[Instrument]': [
            (r'Q[ -]?Exactive[^,\.]*', None),
            (r'Orbitrap[^,\.]*', None),
            (r'LTQ[^,\.]*', None),
            (r'Exploris[^,\.]*', None),
            (r'Fusion[^,\.]*', None),
            (r'timsTOF[^,\.]*', None),
        ],
        'Comment[FragmentationMethod]': [
            (r'HCD', 'HCD'),
            (r'CID', 'CID'),
            (r'ETD', 'ETD'),
            (r'EThcD', 'EThcD'),
        ],
        'Comment[AcquisitionMethod]': [
            (r'DDA|data[- ]dependent', 'DDA'),
            (r'DIA|data[- ]independent', 'DIA'),
            (r'PRM', 'PRM'),
        ],
        'Comment[EnrichmentMethod]': [
            (r'IMAC', 'IMAC'),
            (r'TiO2', 'TiO2'),
            (r'phosphopeptide enrichment', 'phosphopeptide enrichment'),
        ],
        'Characteristics[Label]': [
            (r'TMT[- ]?\d*[NC]?', None),
            (r'iTRAQ[- ]?\d*', None),
            (r'SILAC', 'SILAC'),
            (r'label[- ]?free', 'label free sample'),
        ],
        'Characteristics[OrganismPart]': [
            (r'brain', 'brain'),
            (r'liver', 'liver'),
            (r'kidney', 'kidney'),
            (r'heart', 'heart'),
            (r'lung', 'lung'),
            (r'plasma', 'plasma'),
            (r'serum', 'serum'),
        ],
        'Comment[PrecursorMassTolerance]': [
            (r'\d+\s*ppm\s*(?:precursor|MS1|MS )?(?:mass )?tolerance', None),
            (r'precursor.*?(\d+\s*ppm)', None),
        ],
        'Comment[FragmentMassTolerance]': [
            (r'\d+\.?\d*\s*Da\s*(?:fragment|MS2|MS/MS)?', None),
            (r'fragment.*?(\d+\.?\d*\s*Da)', None),
        ],
    }
    
    def __init__(self):
        self.compiled = {}
        for col, patterns in self.PATTERNS.items():
            self.compiled[col] = [(re.compile(p, re.IGNORECASE), v) for p, v in patterns]
    
    def extract(self, text: str) -> dict:
        """Extract metadata from text."""
        result = {}
        
        for col, patterns in self.compiled.items():
            for pattern, replacement in patterns:
                match = pattern.search(text)
                if match:
                    if replacement:
                        result[col] = replacement
                    else:
                        result[col] = match.group(0).strip()
                    break
        
        return result
    
    def extract_from_filename(self, filename: str) -> dict:
        """Extract metadata from filename."""
        result = {}
        tokens = re.split(r'[_.\\ -]', filename.replace('.raw', ''))
        
        for i, token in enumerate(tokens):
            # Fraction identifiers
            if re.match(r'^[Ff](?:rac)?\d+$', token):
                result['Comment[FractionIdentifier]'] = token
            # Replicate markers
            elif re.match(r'^(?:rep|REP|Rep)\d+$', token):
                result['Characteristics[BiologicalReplicate]'] = token
        
        return result

In [ ]:
def process_manuscript(extractor, pub_data: dict) -> dict:
    """Process manuscript and extract metadata for all raw files."""
    
    # Combine text sections (prioritize Methods)
    text_parts = []
    for section in ['METHODS', 'ABSTRACT', 'TITLE']:
        if section in pub_data and pub_data[section]:
            text_parts.append(pub_data[section])
    
    full_text = '\n'.join(text_parts)
    
    # Extract global metadata
    global_meta = extractor.extract(full_text)
    
    # Get raw files
    raw_files = pub_data.get('Raw Data Files', [])
    
    results = {}
    for raw_file in raw_files:
        file_meta = global_meta.copy()
        file_meta.update(extractor.extract_from_filename(raw_file))
        results[raw_file] = file_meta
    
    return results

In [ ]:
# Run extraction
extractor = SDRFExtractor()
all_predictions = {}

for pxd_id, pub_data in test_data.items():
    print(f"Processing {pxd_id}...")
    all_predictions[pxd_id] = process_manuscript(extractor, pub_data)

print(f"\nProcessed {len(all_predictions)} manuscripts")

In [ ]:
# Create submission
submission = sample_sub.copy()

# Get metadata columns
meta_cols = [c for c in submission.columns 
             if c.startswith(('Characteristics[', 'Comment[', 'FactorValue['))]

# Fill predictions
for idx, row in submission.iterrows():
    pxd = row['PXD']
    raw_file = row['Raw Data File']
    
    if pxd in all_predictions and raw_file in all_predictions[pxd]:
        pred = all_predictions[pxd][raw_file]
        for col in meta_cols:
            submission.at[idx, col] = pred.get(col, 'Not Applicable')
    else:
        for col in meta_cols:
            submission.at[idx, col] = 'Not Applicable'

print(f"Submission shape: {submission.shape}")

In [ ]:
# Check some extractions
print("Sample extractions:")
for col in ['Characteristics[Organism]', 'Characteristics[CleavageAgent]', 'Comment[Instrument]', 'Characteristics[Label]']:
    if col in submission.columns:
        print(f"\n{col}:")
        print(submission[col].value_counts().head(5))

In [ ]:
# Save submission
output_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(output_path, index=False)
print(f"Saved to {output_path}")